In [1]:
from pypdf import PdfReader

reader = PdfReader("../data/apple_10k.pdf")

print(f"Number of pages: {len(reader.pages)}")

first_page_text = reader.pages[0].extract_text()

print(first_page_text[:1000])

Number of pages: 65
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino, California 95014
(Address of principal executive offices) (Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class
Trading
symbol(s) Name of each exchange on which registered
Common Stock, $0.00001 par value per share AAPL The Nasdaq Stock Market

In [2]:
!pip install pypdf

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:

import sys
print(sys.executable)

c:\Users\aryas\AppData\Local\Programs\Python\Python313\python.exe


In [4]:
import sys
!{sys.executable} -m pip install pypdf


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pypdf import PdfReader

companies = {
    "apple": "../data/apple_10k.pdf",
    "microsoft": "../data/microsoft_10k.pdf",
    "tesla": "../data/tesla_10k.pdf"
}

documents = {}  # will store full text per company

for company, path in companies.items():
    reader = PdfReader(path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    documents[company] = full_text
    print(f"{company}: {len(reader.pages)} pages, {len(full_text)} characters extracted")

apple: 65 pages, 206569 characters extracted
microsoft: 138 pages, 374628 characters extracted
tesla: 116 pages, 427191 characters extracted


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\aryas\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

all_chunks = {}

for company in documents:
    chunks = splitter.split_text(documents[company])
    all_chunks[company] = chunks
    print(f"{company}: {len(chunks)} chunks")

apple: 258 chunks
microsoft: 462 chunks
tesla: 535 chunks


In [8]:
!python -m pip install langchain langchain-community


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

apple_chunks = splitter.split_text(documents["apple"])
print(f"Apple 10-K split into {len(apple_chunks)} chunks")
print(apple_chunks[5])

Apple 10-K split into 258 chunks
Indicate by check mark whether the Registrant is a shell company (as defined in Rule 12b-2 of the Act).
Yes  ☐     No  ☒
The aggregate market value of the voting and non-voting stock held by non-affiliates of the Registrant, as of March 28, 2025, the last business day of
the Registrant’s most recently completed second fiscal quarter, was approximately $3,253,431,000,000. Solely for purposes of this disclosure, shares
of common stock held by executive officers and directors of the Registrant as of such date have been excluded because such persons may be
deemed to be affiliates. This determination of executive officers and directors as affiliates is not necessarily a conclusive determination for any other
purposes.
14,776,353,000 shares of common stock were issued and outstanding as of October 17, 2025.
DOCUMENTS INCORPORATED BY REFERENCE


In [10]:
all_chunks = {}  # will store chunks per company

for company in documents:
    chunks = splitter.split_text(documents[company])
    all_chunks[company] = chunks
    print(f"{company}: {len(chunks)} chunks")

apple: 258 chunks
microsoft: 462 chunks
tesla: 535 chunks


In [11]:
!python -m pip install sentence-transformers chromadb


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load the embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create/open ChromaDB
client = chromadb.PersistentClient(path="../data/chroma_db")

collection = client.get_or_create_collection(
    name="financial_10ks"
)

# Add all chunks
for company, chunks in all_chunks.items():

    embeddings = embed_model.encode(chunks).tolist()

    ids = [f"{company}_{i}" for i in range(len(chunks))]

    metadatas = [
        {"company": company, "chunk_index": i}
        for i in range(len(chunks))
    ]

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas
    )

    print(f"Added {len(chunks)} chunks for {company}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1886.04it/s]


Added 258 chunks for apple
Added 462 chunks for microsoft
Added 535 chunks for tesla


In [13]:
# Reconnect to the same ChromaDB
import chromadb

client = chromadb.PersistentClient(path="../data/chroma_db")

collection = client.get_collection(name="financial_10ks")

In [14]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2612.66it/s]


In [15]:
query = "What are Apple's main risk factors?"

query_embedding = embed_model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(meta["company"], f"(distance: {dist:.3f})")
    print(doc[:500])
    print("---")

apple (distance: 0.658)
references to website URLs are intended to be inactive textual references only.
Apple Inc. | 2025 Form 10-K | 4
Item 1A.    Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks.
Statements in this section are based on the Company’s beliefs and opinions regarding matters that could m
---
apple (distance: 0.710)
cause such differences include, but are not limited to, those discussed in Part I, Item 1A of this Form 10-K under the heading “Risk
Factors.” The Company assumes no obligation to revise or update any forward-looking statements for any reason, except as
required by law.
Unless otherwise stated, all information presented herein is based on the Company’s fiscal calendar, and references to particular
years, quarters, months or periods refer to th

In [16]:
!python -m pip install groq


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from groq import Groq

client = Groq(api_key="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")

In [21]:
def generate_answer(query, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    
    prompt = f"""Answer the question using only the context below. 
If the answer isn't in the context, say so — don't make things up.

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)
    
    return response.choices[0].message.content

In [23]:
answer = generate_answer(query, results["documents"][0])
print(answer)

Apple’s main risk factors, as described in the 2025 Form 10‑K, include:

1. **Supply‑chain constraints** – risks of shortages and price increases for components that could materially affect the company’s business, results of operations, financial condition and stock price.  
2. **Design and manufacturing defects** – potential defects in hardware or software that could harm the company’s reputation and financial performance.  
3. **Intense competition** – competitors aggressively cutting prices and lowering margins, with some markets experiencing little or no growth or even contraction.  
4. **Market share and growth** – the company’s minority share in key markets (smartphones, PCs, tablets, wearables) and the possibility of declining demand.  
5. **Product introduction and transition** – the need to successfully launch and transition new products and services to remain competitive.  
6. **Litigation and regulatory changes** – outcomes of investigations, litigation, or changes to laws a

In [29]:
def rag_query_multi_company(question, companies=["apple", "microsoft", "tesla"], n_per_company=3):
    query_embedding = embed_model.encode([question]).tolist()
    
    all_chunks = []
    for company in companies:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=n_per_company,
            where={"company": company}
        )
        all_chunks.extend(results["documents"][0])
    
    context = "\n\n".join(all_chunks)
    prompt = f"""Answer the question using only the context below.
Context:
{context}

Question: {question}

Answer:"""
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [34]:
#checking above function
answer = rag_query("What are Apple's main risk factors?", company="apple")
print(answer)

**Apple’s main risk factors (as described in the 2025 Form 10‑K, Item 1A)**  

- **Supply‑chain constraints** – risk of shortages and price increases for components, especially if Apple shifts to common components rather than customized ones.  
- **Design and manufacturing defects** – potential defects in hardware or software that could harm reputation and business.  
- **Intense competition** – competitors aggressively cutting prices, lowering margins, and in some cases selling at loss, which can erode Apple’s market share and profitability.  
- **Market dynamics** – some markets (smartphones, PCs, tablets, wearables) have little or no growth or may contract, affecting demand.  
- **Inability to compete successfully** – failure to keep up with competitors could materially harm business, reputation, and financial results.  
- **Litigation and regulatory changes** – investigations, lawsuits, or new laws/regulations that require changes to business practices, potentially impacting privac

In [35]:
print(rag_query("What are Microsoft's main risk factors?", company="microsoft"))
print("\n" + "="*80 + "\n")
print(rag_query("What are Tesla's main risk factors?", company="tesla"))
print("\n" + "="*80 + "\n")
print(rag_query("How does Apple's revenue depend on iPhone sales?", company="apple"))

**Microsoft’s main risk factors (as described in the excerpt)**  

1. **Strategic and competitive risks** – uncertainties that could affect the company’s business, operations, financial condition, and stock price.  
2. **Climate‑change and environmental risks** – unclear long‑term effects of climate change, potential new environmental regulations, and changes in energy or resource supply that could raise costs or limit availability of goods and services.  
3. **Global operational and economic risks** – exposure to worldwide economic developments, monetary policy, inflation, recession, and geopolitical events that could adversely impact results.  
4. **Geopolitical and political risks** – political and military disputes that could affect operations and markets.  
5. **Market‑specific risks** –  
   * Growth of the AI PC category.  
   * Attachment of Windows to devices shipped.  
   * Customer mix (consumer, SMB, large enterprise).  
   * Inventory levels in the OEM channel.  
   * Pric

In [36]:
tesla_answer = rag_query("What are Tesla's main risk factors?", company="tesla")
print(tesla_answer)

**Tesla’s main risk factors (as described in the provided excerpt)**  

1. **Limited operating history and customer unfamiliarity** – Tesla’s relatively short track record and the novelty of its products may erode investor confidence.  
2. **Execution risks in scaling operations** – Delays in expanding manufacturing, delivery, and service capabilities could hinder meeting demand.  
3. **Competitive pressures** – Intense competition in the automotive and energy markets, including from established and emerging players.  
4. **Uncertainty about the electric‑vehicle (EV) market** – The future growth of EVs and Tesla’s share of that market is uncertain.  
5. **Quarterly production and sales performance** – Failure to meet market expectations in any reporting period could harm the business.  
6. **Talent acquisition and retention** – Difficulty attracting, hiring, and keeping senior leaders and a large workforce of engineers, technicians, and support staff.  
7. **Negative publicity and crit

In [37]:
answer = rag_query("Which of these companies faces the most regulatory risk?")
print(answer)

I’m sorry, but the context provided does not contain information about multiple companies or a comparison of their regulatory risks, so I can’t determine which company faces the most regulatory risk.


In [38]:
for company in ["apple", "microsoft", "tesla"]:
    results = collection.query(
        query_embeddings=embed_model.encode(["regulatory risk"]).tolist(),
        n_results=3,
        where={"company": company}
    )
    print(company, "chunks retrieved:", len(results["documents"][0]))

apple chunks retrieved: 3
microsoft chunks retrieved: 3
tesla chunks retrieved: 3


In [39]:
#for comparing regulatory risk across companies
def rag_compare(question, companies=["apple", "microsoft", "tesla"], n_per_company=3):
    query_embedding = embed_model.encode([question]).tolist()
    
    labeled_context = ""
    for company in companies:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=n_per_company,
            where={"company": company}
        )
        chunks = results["documents"][0]
        labeled_context += f"\n\n--- {company.upper()} ---\n" + "\n".join(chunks)
    
    prompt = f"""You are given context from three separate companies' SEC filings, clearly labeled by company.
Each company's filing discusses only itself — none of them compare against the others directly.
Your job is to compare them YOURSELF based on what each section says, and give a reasoned answer.
If the context doesn't contain enough information to compare on a specific point, say so for that point specifically — but still compare what you can.

{labeled_context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [40]:
print(rag_compare("Which of these companies faces the most regulatory risk?"))

**Answer: Apple faces the most regulatory risk.**

**Why?**

| Company | Key regulatory‑risk themes in the filing | Breadth / intensity of risk |
|---------|------------------------------------------|-----------------------------|
| **Apple** | • Antitrust investigations and litigation worldwide (U.S., EU, etc.)<br>• Data‑privacy and security obligations (e.g., DMA, EU privacy rules)<br>• New and changing laws/regs on AI, machine‑learning, health & financial services<br>• Supply‑chain safety and industrial‑accident risks<br>• Pandemic‑related restrictions on travel, freight, and consumer demand | Apple’s filing lists **multiple, distinct regulatory arenas**—antitrust, privacy, safety, supply‑chain, public‑health, and emerging tech laws. The company explicitly notes that each of these can materially and adversely affect its business, reputation, and stock price. The sheer number of separate regulatory touchpoints gives Apple the broadest exposure. |
| **Microsoft** | • Competition‑law e